In [5]:
import pandas as pd

from geoai.utils_ds.DataFrameOps import DataFrameOperations
from geoai.utils_ds.PreProcessingOps import PreProcessingOperations
df_ops = DataFrameOperations()
preprocess_ops = PreProcessingOperations()


# READ DF

In [6]:
df = pd.read_csv("csv_files/dataset.csv")
print(df.head(5))
print(df.shape)

       BLUE      GREEN        RED        NIR     SWIR Landcover
0  835.3333   920.7500  1301.0000  1478.6666  1976.75   builtup
1  938.6667   966.3333  1153.0000  1376.0000  2025.20   builtup
2  871.0000   902.0000   946.0000  1218.0000  2025.20   builtup
3  985.0000  1050.0000  1114.6666  1573.0000  1974.75   builtup
4  940.2500   994.5000  1046.0000  1256.5000  1976.75   builtup
(2419, 6)


# SPLIT DATA

Data splitting involves dividing the dataset into separate subsets to train and test the model.
- Training set: Used to fit the model. It learns from this data.
- Test set: Used to assess the performance of the final model. It provides an unbiased evaluation.

In [7]:
X_train, X_test, y_train, y_test= df_ops.split_data(df, "Landcover")
X_train.to_csv("csv_files/X_train.csv", index=False)
X_test.to_csv("csv_files/X_test.csv", index=False)
print(X_train.shape)
print(X_test.shape)

(1935, 5)
(484, 5)


# USING DOMAIN KNOWLEDGE

Feature engineering is the process of creating or improving features. Features are often created based on common sense, domain knowledge, or prior experience. There are certain common techniques for feature creation; however, there is no guarantee that creating new features will improve results. 

For example, we can compute for NDVI, REI, and NDBI which can be derived from raw satellite data and serve as important features that can enhance the analysis and interpretation of remote sensing data.

$$ NDVI = \frac{NIR - RED}{NIR + RED} $$
$$ NDBI = \frac{SWIR - NIR}{SWIR + NIR} $$
$$ REI = \frac{NIR - BLUE}{NIR + BLUE * NIR} $$

In [8]:
X_train["NDVI"] = (X_train["NIR"] - X_train["RED"]) / (X_train["NIR"] + X_train["RED"])
X_train["NDBI"] = (X_train["SWIR"] - X_train["NIR"]) / (X_train["SWIR"] + X_train["NIR"])
X_train["REI"] = (X_train["NIR"] - X_train["BLUE"]) / (X_train["NIR"] + X_train["BLUE"] * X_train["NIR"])
X_train.fillna(0, inplace=True)
X_train.to_csv("csv_files/X_train_with_indices.csv", index=False)

X_test["NDVI"] = (X_test["NIR"] - X_test["RED"]) / (X_test["NIR"] + X_test["RED"])
X_test["NDBI"] = (X_test["SWIR"] - X_test["NIR"]) / (X_test["SWIR"] + X_test["NIR"])
X_test["REI"] = (X_test["NIR"] - X_test["BLUE"]) / (X_test["NIR"] + X_test["BLUE"] * X_test["NIR"])
X_test.fillna(0, inplace=True)
X_test.to_csv("csv_files/X_test_with_indices.csv", index=False)


# Binning/Discretization

 Separate feature values into several bins.

In [9]:
# Create binary NDVI
ndvi_binary_edges = [-float("inf"), 0.5, float("inf")]
ndvi_binary_labels = ["non_veg", "veg"]
X_train,  X_test = preprocess_ops.binarize_or_discretize(X_train, X_test, "NDVI", "NDVI_bin", ndvi_binary_edges, ndvi_binary_labels)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_bin
2402,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,veg


In [10]:
# Create categorical NDVI
ndvi_category_edges = [-float("inf"), 0.2, 0.5, float("inf")]
ndvi_category_labels = ["low_veg", "medium_veg", "high_veg"]
X_train,X_test = preprocess_ops.binarize_or_discretize(
    X_train, X_test, "NDVI", "NDVI_dis", ndvi_category_edges, ndvi_category_labels
)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_bin,NDVI_dis
2402,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,veg,high_veg


# Polynomial transform

In [11]:
numerical_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
print(numerical_columns)
X_train, X_test = preprocess_ops.polynomial_transform(X_train, X_test, numerical_columns, 2)
X_train.head(1)

['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI', 'NDBI', 'REI']


,NDVI_bin,NDVI_dis,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,...,SWIR^2,SWIR NDVI,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2
2402,veg,high_veg,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,...,3964081.0,1538.473561,-413.256318,4.661945,0.597087,-0.160386,0.001809,0.043082,-0.000486,0.000005


In [12]:
X_train.columns 

Index(['NDVI_bin', 'NDVI_dis', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI',
       'NDBI', 'REI', 'BLUE^2', 'BLUE GREEN', 'BLUE RED', 'BLUE NIR',
       'BLUE SWIR', 'BLUE NDVI', 'BLUE NDBI', 'BLUE REI', 'GREEN^2',
       'GREEN RED', 'GREEN NIR', 'GREEN SWIR', 'GREEN NDVI', 'GREEN NDBI',
       'GREEN REI', 'RED^2', 'RED NIR', 'RED SWIR', 'RED NDVI', 'RED NDBI',
       'RED REI', 'NIR^2', 'NIR SWIR', 'NIR NDVI', 'NIR NDBI', 'NIR REI',
       'SWIR^2', 'SWIR NDVI', 'SWIR NDBI', 'SWIR REI', 'NDVI^2', 'NDVI NDBI',
       'NDVI REI', 'NDBI^2', 'NDBI REI', 'REI^2'],
      dtype='object')

# APPLY ONE HOT ENCODING

Process used to convert categorical data into a binary (0,1) vector representation where each category is represented by a unique vector in the space.

In [13]:
X_train, X_test = preprocess_ops.make_one_hot_encoder(X_train, X_test, "NDVI_bin")
X_train.head(1)

,NDVI_dis,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,...,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin_non_veg,NDVI_bin_veg
2402,high_veg,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,139502.25,...,-413.256318,4.661945,0.597087,-0.160386,0.001809,0.043082,-0.000486,0.000005,0,1


# MAKE ORDINAL ENCODER

In [14]:
categories = [["low_veg", "medium_veg", "high_veg"]]
X_train, X_test = preprocess_ops.make_ordinal_encoder(X_train, X_test, "NDVI_dis", categories)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin_non_veg,NDVI_bin_veg,NDVI_dis_encoded
2402,373.5,519.5,389.0,3034.0,1991.0,0.772714,-0.207562,0.002342,139502.25,194033.25,...,4.661945,0.597087,-0.160386,0.001809,0.043082,-0.000486,0.000005,0,1,2


In [15]:
# Export
X_train.to_csv("csv_files/X_train_fe.csv", index=False)
X_test.to_csv("csv_files/X_test_fe.csv", index=False)

# CONVERT LABEL STRING TO INTEGERS

In [16]:
y_train, y_test = preprocess_ops.make_label_encoder(y_train, y_test)
y_train.to_csv("csv_files/y_train.csv", index=False)
y_test.to_csv("csv_files/y_test.csv", index=False)

{'builtup': 0, 'grass': 1, 'road': 2, 'trees': 3}


END